# Transformer Math Complete

Part of **ML Math Applied**.

## 1. Intuition First

A transformer block combines attention, residual paths, and normalization so token information can mix without becoming numerically unstable.

![Transformer block](../assets/diagrams/transformer_block.svg)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5)

## 2. Block Equations

For token states $X$:

$$
\mathrm{Attention}(X) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V
$$

where $M$ is the causal mask. A transformer block then applies

$$
H_1 = \mathrm{LayerNorm}(X + \mathrm{Attention}(X)),
\qquad
H_2 = \mathrm{LayerNorm}(H_1 + \mathrm{FFN}(H_1)).
$$

Residual adds preserve the incoming token state while the new sublayer learns a correction.

In [ ]:
from src.ml_components import causal_attention_mask, layer_norm_forward, scaled_dot_product_attention

def gelu_numpy(values: np.ndarray) -> np.ndarray:
    return 0.5 * values * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (values + 0.044715 * values**3)))

tokens = np.array([[[1.0, 0.3, -0.2], [0.8, 0.4, 0.1], [0.2, 1.0, 0.6], [0.0, 0.7, 1.1]]])
W_q = np.array([[0.8, 0.1, 0.0], [0.2, 0.7, 0.1], [0.0, 0.3, 0.9]])
W_k = np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1], [0.0, 0.2, 0.9]])
W_v = np.array([[0.6, 0.0, 0.1], [0.2, 0.8, 0.0], [0.1, 0.1, 0.7]])
W1 = np.array([[0.5, -0.2, 0.3, 0.1], [0.2, 0.6, -0.1, 0.4], [0.3, 0.1, 0.7, -0.2]])
b1 = np.array([0.0, -0.1, 0.2, 0.05])
W2 = np.array([[0.6, 0.1, -0.2], [0.2, 0.5, 0.1], [-0.1, 0.3, 0.6], [0.4, -0.2, 0.2]])
b2 = np.array([0.05, -0.05, 0.1])

Q = tokens @ W_q
K = tokens @ W_k
V = tokens @ W_v
mask = causal_attention_mask(tokens.shape[1])[None, :, :]

attention_output, attention_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
residual1 = tokens + attention_output
norm1, _ = layer_norm_forward(residual1)
ff_hidden = gelu_numpy(norm1 @ W1 + b1)
ff_output = ff_hidden @ W2 + b2
block_output, _ = layer_norm_forward(norm1 + ff_output)

print("last token after block =", block_output[0, -1])
print("attention for last token =", attention_weights[0, -1])

## 3. PyTorch Verification

Rebuild the same masked-attention and feed-forward block in PyTorch.

In [ ]:
tokens_t = torch.tensor(tokens, dtype=torch.float64)
W_q_t = torch.tensor(W_q, dtype=torch.float64)
W_k_t = torch.tensor(W_k, dtype=torch.float64)
W_v_t = torch.tensor(W_v, dtype=torch.float64)
W1_t = torch.tensor(W1, dtype=torch.float64)
b1_t = torch.tensor(b1, dtype=torch.float64)
W2_t = torch.tensor(W2, dtype=torch.float64)
b2_t = torch.tensor(b2, dtype=torch.float64)
mask_t = torch.tensor(mask)

Q_t = tokens_t @ W_q_t
K_t = tokens_t @ W_k_t
V_t = tokens_t @ W_v_t
scores_t = Q_t @ K_t.transpose(-1, -2) / np.sqrt(tokens.shape[-1])
scores_t = torch.where(mask_t, scores_t, torch.full_like(scores_t, -1e9))
attention_output_t = torch.softmax(scores_t, dim=-1) @ V_t
residual1_t = tokens_t + attention_output_t
norm1_t = torch.nn.functional.layer_norm(residual1_t, normalized_shape=(tokens.shape[-1],))
ff_hidden_t = torch.nn.functional.gelu(norm1_t @ W1_t + b1_t, approximate="tanh")
ff_output_t = ff_hidden_t @ W2_t + b2_t
block_output_t = torch.nn.functional.layer_norm(norm1_t + ff_output_t, normalized_shape=(tokens.shape[-1],))

print(torch.allclose(block_output_t, torch.tensor(block_output), atol=1e-6))

## 4. Custom Figure

The block changes token norms while the attention map shows how much each token borrows from previous positions.

In [ ]:
token_norms_before = np.linalg.norm(tokens[0], axis=-1)
token_norms_after = np.linalg.norm(block_output[0], axis=-1)
labels = [f"t{i}" for i in range(tokens.shape[1])]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
heatmap = axes[0].imshow(attention_weights[0], cmap="Blues", vmin=0.0, vmax=1.0)
axes[0].set_xticks(range(len(labels)), labels)
axes[0].set_yticks(range(len(labels)), labels)
axes[0].set_title("Masked attention matrix")

width = 0.35
positions = np.arange(len(labels))
axes[1].bar(positions - width / 2, token_norms_before, width=width, label="before")
axes[1].bar(positions + width / 2, token_norms_after, width=width, label="after")
axes[1].set_xticks(positions, labels)
axes[1].set_title("Token-state norms through the block")
axes[1].legend()

fig.colorbar(heatmap, ax=axes[0], fraction=0.046)
plt.tight_layout()
plt.show()

## 5. Case Study: Why Residual Paths Matter

If attention temporarily focuses on the wrong token, the residual path still preserves the pre-attention state.
That makes optimization much easier than forcing every layer to rebuild information from scratch.